# Transactions

## Quick notes
### What this covers
How to group several statements into one all-or-nothing unit (a transaction), how SQLite handles locks and concurrency, and the classic mistakes (lost updates, half-finished work).
### Key points
- A transaction is a group of statements that either ALL happen (`COMMIT`) or NONE happen (`ROLLBACK`).
- ACID: Atomic (all or nothing), Consistent (rules always hold), Isolated (others do not see half-done work), Durable (committed means saved).
- Without `BEGIN`, every single statement is its own mini transaction (autocommit).
- `BEGIN DEFERRED` (default) takes locks late, `BEGIN IMMEDIATE` takes the write lock now, `BEGIN EXCLUSIVE` blocks readers too (in rollback-journal mode).
- `SAVEPOINT name` / `ROLLBACK TO name` / `RELEASE name` give you nested "undo points".
- SQLite allows ONE writer at a time. WAL mode lets readers keep reading while one writer writes.
- Do read-modify-write in ONE statement (`UPDATE ... SET x = x + 1`), not read in app code then write back.
### Gotchas (the part worth revising twice)
- An error inside a transaction does not always roll it back. By default only the failing STATEMENT is undone; the transaction stays open and you must `ROLLBACK` or `COMMIT`.
- DDL (`CREATE`, `DROP`, `ALTER`) is transactional in SQLite and can be rolled back (unlike MySQL).
- Forgetting `COMMIT` leaves the transaction open and holds locks. Other connections get "database is locked".
- Python's `sqlite3` module may open transactions for you (see `isolation_level`); this notebook uses autocommit so what you write is what runs.
- `PRAGMA foreign_keys` cannot be changed inside a transaction (it is silently ignored).
### When it matters
Money transfers, stock changes, multi-table inserts, schema migrations, anything where "half done" would be a bug.
### One-line recap example
```sql
BEGIN IMMEDIATE;
UPDATE acct SET bal = bal - 100 WHERE id = 1;
UPDATE acct SET bal = bal + 100 WHERE id = 2;
COMMIT;   -- or ROLLBACK; to undo both
```

## 01. ACID in plain words
- **Atomic**: a bank transfer is two updates. Either both happen or neither. Never "money left account A but did not arrive in B".
- **Consistent**: after the transaction, all rules (constraints, foreign keys, CHECKs) still hold.
- **Isolated**: while you are halfway through, other users do not see your half-done state.
- **Durable**: after `COMMIT` returns, the data survives a crash or power cut.
- In SQLite the journal (rollback journal or WAL file) is what makes atomic + durable work.

## 02. Seed data
- Two small tables used throughout: bank `accounts` and a `ledger` of transfers.
- `CHECK (balance >= 0)` is our safety net: an overdraft is a constraint violation.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE accounts (
    id      INTEGER PRIMARY KEY,
    owner   TEXT NOT NULL,
    balance INTEGER NOT NULL CHECK (balance >= 0)   -- cents
);
CREATE TABLE ledger (
    id      INTEGER PRIMARY KEY,
    from_id INTEGER NOT NULL REFERENCES accounts(id),
    to_id   INTEGER NOT NULL REFERENCES accounts(id),
    amount  INTEGER NOT NULL CHECK (amount > 0)
);
INSERT INTO accounts (id, owner, balance) VALUES (1, 'Asha', 10000), (2, 'Ben', 5000), (3, 'Chitra', 0);
SELECT * FROM accounts ORDER BY id;

id | owner  | balance
---+--------+--------
1  | Asha   | 10000  
2  | Ben    | 5000   
3  | Chitra | 0      
(3 rows)



## 03. Autocommit: every statement is its own transaction
- Outside `BEGIN`, each statement commits immediately and permanently.
- That is why a two-step job (debit, then credit) is dangerous without a transaction: a crash between the steps loses money.
- SQL itself cannot tell you if a transaction is open; Python's `con.in_transaction` can (used in section 06).

In [3]:
%%sql
-- Without a transaction: the debit is already permanent even if the credit never runs
UPDATE accounts SET balance = balance - 1000 WHERE id = 1;
SELECT id, owner, balance FROM accounts WHERE id IN (1, 2) ORDER BY id;

id | owner | balance
---+-------+--------
1  | Asha  | 9000   
2  | Ben   | 5000   
(2 rows)



In [4]:
%%sql
-- put the money back so later examples start clean
UPDATE accounts SET balance = balance + 1000 WHERE id = 1;
SELECT id, owner, balance FROM accounts ORDER BY id;

id | owner  | balance
---+--------+--------
1  | Asha   | 10000  
2  | Ben    | 5000   
3  | Chitra | 0      
(3 rows)



## 04. BEGIN, COMMIT, ROLLBACK
- `BEGIN;` starts. `COMMIT;` (or `END;`) makes everything permanent. `ROLLBACK;` throws everything away.
- Inside the transaction YOU see your own changes straight away.
- After `ROLLBACK` the table is exactly as it was at `BEGIN`.

In [5]:
%%sql
BEGIN;
UPDATE accounts SET balance = balance - 2000 WHERE id = 1;
UPDATE accounts SET balance = balance + 2000 WHERE id = 2;
SELECT id, owner, balance FROM accounts WHERE id IN (1, 2) ORDER BY id;   -- my own view, mid-transaction
ROLLBACK;
SELECT id, owner, balance FROM accounts WHERE id IN (1, 2) ORDER BY id;   -- back to normal

id | owner | balance
---+-------+--------
1  | Asha  | 8000   
2  | Ben   | 7000   
(2 rows)

id | owner | balance
---+-------+--------
1  | Asha  | 10000  
2  | Ben   | 5000   
(2 rows)



## 05. An atomic transfer
- The classic example: debit one account, credit another, write a ledger row. All three or nothing.
- Here it succeeds, so we `COMMIT`.
- In a real app you wrap this in try/except: `COMMIT` on success, `ROLLBACK` on any exception.

In [6]:
%%sql
BEGIN IMMEDIATE;
UPDATE accounts SET balance = balance - 2500 WHERE id = 1;
UPDATE accounts SET balance = balance + 2500 WHERE id = 2;
INSERT INTO ledger (from_id, to_id, amount) VALUES (1, 2, 2500);
COMMIT;
SELECT id, owner, balance FROM accounts ORDER BY id;
SELECT * FROM ledger ORDER BY id;

id | owner  | balance
---+--------+--------
1  | Asha   | 7500   
2  | Ben    | 7500   
3  | Chitra | 0      
(3 rows)

id | from_id | to_id | amount
---+---------+-------+-------
1  | 1       | 2     | 2500  
(1 row)



## 06. What happens when a statement fails inside a transaction
- Default behaviour (`ABORT`): only the failing STATEMENT is undone. Earlier statements in the transaction stay, and the transaction is still open.
- So after an error you must decide: `ROLLBACK` everything, or carry on and `COMMIT` the good parts. Forgetting to do either leaves the transaction hanging.
- Below, the debit works, then the credit step fails a constraint. If we `COMMIT` now, we would keep the debit with no matching credit: money vanished.
- Gotcha for Python: an uncaught exception does not roll back for you unless you use `with con:` or call `rollback()`.

In [7]:
%%sql expect_error
BEGIN;
UPDATE accounts SET balance = balance - 4000 WHERE id = 2;       -- Ben: 7500 -> 3500 (ok)
UPDATE accounts SET balance = balance - 9000 WHERE id = 3;       -- Chitra has 0: CHECK fails
SELECT id, owner, balance FROM accounts ORDER BY id;             -- debit is still there, txn still open!

ERROR: CHECK constraint failed: balance >= 0

id | owner  | balance
---+--------+--------
1  | Asha   | 7500   
2  | Ben    | 3500   
3  | Chitra | 0      
(3 rows)



In [8]:
print("still inside a transaction?", con.in_transaction)
con.execute("ROLLBACK")
print("after ROLLBACK:", con.in_transaction)

still inside a transaction? True
after ROLLBACK: False


In [9]:
%%sql
SELECT id, owner, balance FROM accounts ORDER BY id;   -- Ben is back to 7500

id | owner  | balance
---+--------+--------
1  | Asha   | 7500   
2  | Ben    | 7500   
3  | Chitra | 0      
(3 rows)



## 07. Error handling pattern: all or nothing in Python
- Wrap the transaction in try/except and roll back on any exception.
- `con.execute("BEGIN")` ... `con.execute("COMMIT")` in `try`, `con.execute("ROLLBACK")` in `except`.
- `with con:` on a normal (non-autocommit) connection does the same automatically.
- Here: a transfer function that refuses to overdraw.

In [10]:
import sqlite3
def transfer(src, dst, amount):
    try:
        con.execute("BEGIN IMMEDIATE")
        con.execute("UPDATE accounts SET balance = balance - ? WHERE id = ?", (amount, src))
        con.execute("UPDATE accounts SET balance = balance + ? WHERE id = ?", (amount, dst))
        con.execute("INSERT INTO ledger (from_id, to_id, amount) VALUES (?, ?, ?)", (src, dst, amount))
        con.execute("COMMIT")
        return "ok"
    except sqlite3.Error as exc:
        con.execute("ROLLBACK")
        return f"rolled back: {exc}"

print(transfer(2, 3, 1000))     # fine
print(transfer(3, 1, 999999))   # Chitra cannot afford it: whole thing undone
print(con.execute("SELECT id, balance FROM accounts ORDER BY id").fetchall())

ok
rolled back: CHECK constraint failed: balance >= 0
[(1, 7500), (2, 6500), (3, 1000)]


## 08. Constraint violations mid-transaction and conflict clauses
- `OR ROLLBACK` on a statement: if it fails, the WHOLE transaction rolls back immediately.
- `OR ABORT` (default): undo this statement only. `OR FAIL`: keep rows already changed by this same statement, then stop. `OR IGNORE`: skip bad rows.
- Pick `OR ROLLBACK` when any failure means the whole unit is invalid.

In [11]:
%%sql expect_error
BEGIN;
UPDATE accounts SET balance = balance + 100 WHERE id = 1;
UPDATE OR ROLLBACK accounts SET balance = balance - 99999 WHERE id = 1;   -- fails, rolls back everything
SELECT id, balance FROM accounts WHERE id = 1;                            -- Asha's +100 is gone too

ERROR: CHECK constraint failed: balance >= 0

id | balance
---+--------
1  | 7500   
(1 row)



In [12]:
print("in transaction after OR ROLLBACK failure?", con.in_transaction)

in transaction after OR ROLLBACK failure? False


## 09. DDL is transactional
- In SQLite you can `CREATE TABLE`, `ALTER TABLE`, `DROP TABLE` inside a transaction and roll it back.
- This makes schema migrations safe: run all steps in one transaction, and if one fails, nothing changes.
- (MySQL commits DDL automatically; PostgreSQL is transactional like SQLite.)

In [13]:
%%sql
BEGIN;
CREATE TABLE temp_experiment (id INTEGER PRIMARY KEY);
ALTER TABLE accounts ADD COLUMN nickname TEXT;
SELECT count(*) AS tables_named_temp_experiment FROM sqlite_master WHERE name = 'temp_experiment';
ROLLBACK;
SELECT count(*) AS tables_named_temp_experiment FROM sqlite_master WHERE name = 'temp_experiment';
SELECT name FROM pragma_table_info('accounts') ORDER BY cid;   -- no nickname column

tables_named_temp_experiment
----------------------------
1                           
(1 row)

tables_named_temp_experiment
----------------------------
0                           
(1 row)

name   
-------
id     
owner  
balance
(3 rows)



## 10. SAVEPOINT, RELEASE, ROLLBACK TO
- A savepoint is a named undo point inside a transaction (they can nest).
- `ROLLBACK TO name` undoes work back to that point; the savepoint and the outer transaction stay open.
- `RELEASE name` forgets the savepoint, keeping its changes (merged into the parent).
- Use case: try an optional step; if it fails, undo only that step and carry on.
- Gotcha: `ROLLBACK TO` does NOT end the transaction. You still need `COMMIT`.

In [14]:
%%sql
BEGIN;
UPDATE accounts SET balance = balance + 100 WHERE id = 3;          -- step 1: keep
SAVEPOINT bonus;
UPDATE accounts SET balance = balance + 5000 WHERE id = 3;         -- step 2: experimental
SELECT id, balance FROM accounts WHERE id = 3;
ROLLBACK TO bonus;                                                 -- undo only step 2
SELECT id, balance FROM accounts WHERE id = 3;
RELEASE bonus;
COMMIT;
SELECT id, owner, balance FROM accounts ORDER BY id;

id | balance
---+--------
3  | 6100   
(1 row)

id | balance
---+--------
3  | 1100   
(1 row)

id | owner  | balance
---+--------+--------
1  | Asha   | 7500   
2  | Ben    | 6500   
3  | Chitra | 1100   
(3 rows)



In [15]:
%%sql
-- Nested savepoints: roll back the inner one, keep the outer one
BEGIN;
SAVEPOINT outer_sp;
UPDATE accounts SET owner = 'Asha K' WHERE id = 1;
SAVEPOINT inner_sp;
UPDATE accounts SET owner = 'Oops' WHERE id = 1;
ROLLBACK TO inner_sp;
RELEASE outer_sp;       -- releases outer (and inner) and keeps the 'Asha K' change
COMMIT;
SELECT id, owner FROM accounts WHERE id = 1;

id | owner 
---+-------
1  | Asha K
(1 row)



## 11. BEGIN DEFERRED vs IMMEDIATE vs EXCLUSIVE
- `BEGIN` / `BEGIN DEFERRED`: takes no lock until the first read (shared lock) or first write (write lock). Another connection may grab the write lock first and make your later write fail with SQLITE_BUSY.
- `BEGIN IMMEDIATE`: takes the write lock right now. If you plan to write, use this, so you fail early (at BEGIN) not half way.
- `BEGIN EXCLUSIVE`: like IMMEDIATE, and in rollback-journal mode also stops other connections reading. In WAL mode it behaves like IMMEDIATE.
- A pure read-only transaction can stay `DEFERRED`: it gives a consistent snapshot to read from.

In [16]:
%%sql
BEGIN IMMEDIATE;
SELECT 'holding the write lock now' AS note;
COMMIT;
BEGIN EXCLUSIVE;
SELECT 'exclusive lock held' AS note;
COMMIT;

note                      
--------------------------
holding the write lock now
(1 row)

note               
-------------------
exclusive lock held
(1 row)



## 12. Isolation and locking in SQLite
- SQLite is serialisable: transactions behave as if they ran one after another.
- Only ONE connection can write at a time (database-wide write lock). Many can read.
- **Rollback journal** (default): a writer needs readers to finish before it can commit. Readers and writers block each other.
- **WAL mode** (`PRAGMA journal_mode = WAL`): writes go to a separate log. Readers keep reading the old snapshot while a writer works. Still only one writer. Great for web apps.
- WAL needs a real file database; it cannot apply to `:memory:`. The next cell shows that honestly.

In [17]:
%%sql
PRAGMA journal_mode;            -- in-memory databases report 'memory'
PRAGMA journal_mode = WAL;      -- request WAL: an in-memory database cannot switch, it just reports what it is using

journal_mode
------------
memory      
(1 row)

journal_mode
------------
memory      
(1 row)



## 13. Seeing locks in action with two connections
- To show real locking we need a file database and two connections. The code uses a temporary folder and throws it away; its path is not printed.
- Connection A starts a write transaction and does not commit. Connection B then tries to write.
- With `timeout=0` (no waiting) B fails at once with "database is locked". This is the SQLITE_BUSY error.
- `PRAGMA busy_timeout = ms` (or the `timeout` argument in Python) makes B wait and retry for that long before failing.

In [18]:
import tempfile, os, sqlite3
with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, "demo.db")
    a = sqlite3.connect(path, isolation_level=None, timeout=0)
    b = sqlite3.connect(path, isolation_level=None, timeout=0)
    a.execute("CREATE TABLE t (id INTEGER PRIMARY KEY, n INTEGER)")
    a.execute("INSERT INTO t VALUES (1, 0)")
    print("journal mode of the file db:", a.execute("PRAGMA journal_mode").fetchone()[0])

    a.execute("BEGIN IMMEDIATE")                     # A takes the write lock
    a.execute("UPDATE t SET n = n + 1 WHERE id = 1")
    try:
        b.execute("BEGIN IMMEDIATE")                 # B wants it too
    except sqlite3.OperationalError as exc:
        print("B could not start writing:", exc)
    print("rollback-journal mode: B can still read while A has only a pending write:", b.execute("SELECT n FROM t").fetchone()[0])
    a.execute("COMMIT")
    b.execute("BEGIN IMMEDIATE")                     # now it works
    b.execute("UPDATE t SET n = n + 1 WHERE id = 1")
    b.execute("COMMIT")
    print("final n:", a.execute("SELECT n FROM t").fetchone()[0])

    print("switching to WAL:", a.execute("PRAGMA journal_mode = WAL").fetchone()[0])
    a.execute("BEGIN IMMEDIATE")
    a.execute("UPDATE t SET n = 100 WHERE id = 1")
    print("WAL: B reads while A is mid-write, sees committed value:", b.execute("SELECT n FROM t").fetchone()[0])
    a.execute("COMMIT")
    print("WAL: after A commits, B sees:", b.execute("SELECT n FROM t").fetchone()[0])
    a.close(); b.close()

journal mode of the file db: delete
B could not start writing: database is locked
rollback-journal mode: B can still read while A has only a pending write: 0
final n: 2
switching to WAL: wal
WAL: B reads while A is mid-write, sees committed value: 2
WAL: after A commits, B sees: 100


## 14. Busy, "database is locked", and deadlocks
- SQLite has no classic multi-row deadlock detector. Instead you get `SQLITE_BUSY` ("database is locked").
- A real deadlock-like case: two DEFERRED transactions both read, then both try to write. One must give up (busy). Fix: use `BEGIN IMMEDIATE` for any transaction that will write.
- Set `PRAGMA busy_timeout = 5000;` (5 seconds) on every connection so short clashes just wait.
- Keep transactions SHORT: never wait for a human, a network call or `input()` while holding a transaction open.
- On a busy error: roll back, wait a little, retry the whole transaction.

In [19]:
%%sql
PRAGMA busy_timeout = 5000;
PRAGMA busy_timeout;

timeout
-------
5000   
(1 row)

timeout
-------
5000   
(1 row)



## 15. Deferred foreign keys
- Normally a foreign key is checked after EACH statement. Sometimes two rows reference each other, or you load child rows before parents.
- `DEFERRABLE INITIALLY DEFERRED` on the foreign key delays the check until `COMMIT`. If it is still broken at commit, the commit fails.
- `PRAGMA defer_foreign_keys = ON` does the same for the current transaction only (resets at the end).
- See 05-DDL-And-Integrity/01-Constraints-And-Keys for foreign key basics.

In [20]:
%%sql
CREATE TABLE team (id INTEGER PRIMARY KEY, name TEXT, lead_id INTEGER REFERENCES person(id) DEFERRABLE INITIALLY DEFERRED);
CREATE TABLE person (id INTEGER PRIMARY KEY, name TEXT, team_id INTEGER REFERENCES team(id) DEFERRABLE INITIALLY DEFERRED);
BEGIN;
INSERT INTO team (id, name, lead_id) VALUES (1, 'Data', 10);   -- person 10 does not exist yet: allowed for now
INSERT INTO person (id, name, team_id) VALUES (10, 'Asha', 1);
COMMIT;
SELECT t.name AS team, p.name AS lead FROM team t JOIN person p ON p.id = t.lead_id ORDER BY t.id;

team | lead
-----+-----
Data | Asha
(1 row)



In [21]:
%%sql expect_error
-- If the problem is still there at COMMIT, the COMMIT fails and the transaction stays open: you must ROLLBACK
BEGIN;
INSERT INTO team (id, name, lead_id) VALUES (2, 'Ghost', 99);    -- person 99 never arrives
COMMIT;

ERROR: FOREIGN KEY constraint failed



In [22]:
print("transaction still open after failed COMMIT?", con.in_transaction)
con.execute("ROLLBACK")

transaction still open after failed COMMIT? True


## 16. Lost update: the read-modify-write trap
- Bad pattern: app reads a value, adds to it in code, writes it back. If two users do this at the same time, one change is lost.
- Both read 10, both compute 11, both write 11. Final 11, should be 12.
- Fix 1 (best): one statement, `UPDATE ... SET n = n + 1`. The database does read + write atomically.
- Fix 2: `BEGIN IMMEDIATE` around the read and the write so nobody else can write in between.
- Fix 3: optimistic locking, a `version` column: `UPDATE ... WHERE id = ? AND version = ?`, and check `changes()` is 1.

In [23]:
%%sql
CREATE TABLE counters (id INTEGER PRIMARY KEY, name TEXT, n INTEGER NOT NULL, version INTEGER NOT NULL DEFAULT 0);
INSERT INTO counters (id, name, n) VALUES (1, 'page_views', 10);
SELECT * FROM counters;

id | name       | n  | version
---+------------+----+--------
1  | page_views | 10 | 0      
(1 row)



In [24]:
# Two "users" read at the same time, then each writes back its own result (bad)
r1 = con.execute("SELECT n FROM counters WHERE id = 1").fetchone()[0]
r2 = con.execute("SELECT n FROM counters WHERE id = 1").fetchone()[0]
con.execute("UPDATE counters SET n = ? WHERE id = 1", (r1 + 1,))
con.execute("UPDATE counters SET n = ? WHERE id = 1", (r2 + 1,))
print("lost update, expected 12, got:", con.execute("SELECT n FROM counters WHERE id = 1").fetchone()[0])

lost update, expected 12, got: 11


In [25]:
%%sql
-- Good: the database does the arithmetic, so no update can be lost
UPDATE counters SET n = 10 WHERE id = 1;
UPDATE counters SET n = n + 1 WHERE id = 1;
UPDATE counters SET n = n + 1 WHERE id = 1;
SELECT n AS expected_12 FROM counters WHERE id = 1;

expected_12
-----------
12         
(1 row)



In [26]:
%%sql
-- Optimistic locking: the second writer used a stale version, so it matches 0 rows
UPDATE counters SET n = 50, version = version + 1 WHERE id = 1 AND version = 0;   -- first writer wins
SELECT changes() AS rows_changed_by_first;
UPDATE counters SET n = 99, version = version + 1 WHERE id = 1 AND version = 0;   -- stale version
SELECT changes() AS rows_changed_by_stale;
SELECT n, version FROM counters WHERE id = 1;

rows_changed_by_first
---------------------
1                    
(1 row)

rows_changed_by_stale
---------------------
0                    
(1 row)

n  | version
---+--------
50 | 1      
(1 row)



## 17. Check-then-act race and the single-statement fix
- Bad: `SELECT stock` then, if enough, `UPDATE stock = stock - qty`. Between the two statements someone else may take the last item.
- Good: put the check inside the UPDATE's WHERE and check `changes()`. One statement, atomic.
- Same idea for "reserve a seat", "use a coupon once", "withdraw if funds".

In [27]:
%%sql
CREATE TABLE stock_item (sku TEXT PRIMARY KEY, qty INTEGER NOT NULL CHECK (qty >= 0));
INSERT INTO stock_item VALUES ('SKU-1', 3);
-- ask for 2: succeeds
UPDATE stock_item SET qty = qty - 2 WHERE sku = 'SKU-1' AND qty >= 2;
SELECT changes() AS reserved_2;
-- ask for 2 again: only 1 left, so 0 rows change and we know it failed, with no error and no race
UPDATE stock_item SET qty = qty - 2 WHERE sku = 'SKU-1' AND qty >= 2;
SELECT changes() AS reserved_2_again;
SELECT * FROM stock_item;

reserved_2
----------
1         
(1 row)

reserved_2_again
----------------
0               
(1 row)

sku   | qty
------+----
SKU-1 | 1  
(1 row)



## 18. Idempotent operations
- Idempotent = running it twice has the same effect as running it once. Vital when you retry after a failure or timeout.
- Make inserts idempotent with a unique key plus `ON CONFLICT DO NOTHING`, or an UPSERT.
- (In the example, `changes() = 1` is a compact trick; in app code, prefer `INSERT ... RETURNING` and check whether a row came back.)
- Example: a payment with a unique `payment_ref`. If the network drops and the client retries, the second insert is ignored instead of charging twice.
- `UPDATE t SET status = 'paid'` is idempotent; `UPDATE t SET amount = amount + 10` is not.

In [28]:
%%sql
CREATE TABLE payments (ref TEXT PRIMARY KEY, account_id INTEGER NOT NULL, amount INTEGER NOT NULL);
BEGIN;
INSERT INTO payments VALUES ('PAY-001', 1, 700) ON CONFLICT (ref) DO NOTHING;
UPDATE accounts SET balance = balance - 700 WHERE id = 1 AND changes() = 1;   -- only if the payment row was really new
COMMIT;
-- retry the exact same request: no second charge
BEGIN;
INSERT INTO payments VALUES ('PAY-001', 1, 700) ON CONFLICT (ref) DO NOTHING;
UPDATE accounts SET balance = balance - 700 WHERE id = 1 AND changes() = 1;
COMMIT;
SELECT id, balance FROM accounts WHERE id = 1;
SELECT count(*) AS payment_rows FROM payments;

id | balance
---+--------
1  | 6800   
(1 row)

payment_rows
------------
1           
(1 row)



## 19. Durability knobs and a short checklist
- `PRAGMA synchronous`: FULL (safest), NORMAL (fine with WAL), OFF (fast, can corrupt on power loss).
- One big transaction is MUCH faster than thousands of autocommit inserts, because the disk sync happens once.
- Checklist: use `BEGIN IMMEDIATE` for writers; keep transactions short; set `busy_timeout`; always handle the error path with `ROLLBACK`; do arithmetic in SQL; make retries idempotent.
- See 11 / 01-Data-Modification for UPSERT and RETURNING.

In [29]:
%%sql
PRAGMA synchronous;

synchronous
-----------
2          
(1 row)

